# Part 4: second dataset, rel-stack (Stack Exchange)

**Just do Runtime → Run all** (L4 GPU, High-RAM). About **4–5 hours**; it can run in the background. If it stops, run the first 5 cells, then the **Resume** cell.

Task `user-engagement`: will a user post, comment or vote in the next 3 months? Users are labelled every 3 months, so each user has their own label history, one hop from the user in the graph. We train on the most recent ~250K training rows (validation and test are complete); label history still uses every training label.

In [ ]:
REPO_URL = "https://github.com/mbermad/relational-trials.git"
import os, glob, shutil
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cs224w-relational-trials"
OUT = f"{WORK}/results_stack"
CACHE = "/content/cache"  # on the Colab disk, not Drive
os.makedirs(OUT, exist_ok=True)
TASKS = "user-engagement"
COMMON = f"--dataset rel-stack --tasks {TASKS} --max_train 250000 --seeds 0 1 2 --cache_dir {CACHE} --results {OUT}"
GNN = "--text glove --epochs 30 --patience 3 --max_steps_per_epoch 500"

In [ ]:
%cd /content
!rm -rf relational-trials && git clone -q {REPO_URL}
%cd /content/relational-trials
!git log --oneline -1
!pip install -q -e . torch_geometric pytorch_frame sentence-transformers

In [ ]:
import torch
TORCH = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"  # pyg-lib wheels are built per minor version
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
import pyg_lib, psutil
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      f"| RAM: {psutil.virtual_memory().total / 2**30:.0f} GB")

Tests (1–2 minutes). **Expect 29 passed.** If anything fails, stop and send the error.

In [ ]:
!pytest -q

## 1. Explore + LightGBM (CPU, about 30–45 minutes; downloads the data first)

In [ ]:
!python scripts/run.py explore {COMMON}
!python scripts/run.py lgbm --features table agg agg_nohist {COMMON}

## 2. GNN with text, without and with label history (about 2–3 hours; the first run also embeds all text, ~20–40 min)

In [ ]:
!python scripts/run.py rdl {GNN} {COMMON}
!python scripts/run.py rdl {GNN} --history {COMMON}

## Resume: run only what is missing
If the session stopped, run the first 5 cells and then this one.

In [ ]:
STEPS = {"lgbm_table": "lgbm --features table", "lgbm_agg": "lgbm --features agg",
         "lgbm_agg_nohist": "lgbm --features agg_nohist",
         "rdl_sage": f"rdl {GNN}", "rdl_sage_hist": f"rdl {GNN} --history"}
for task in TASKS.split():
    for name, cmd in STEPS.items():
        if os.path.exists(f"{OUT}/runs/{task}/{name}/seed2.json"):
            print(f"done     {task:16s} {name}"); continue
        print(f"running  {task:16s} {name}")
        !python scripts/run.py {cmd} {COMMON.replace(f"--tasks {TASKS}", f"--tasks {task}")}

## 3. Results, significance, download

In [ ]:
from IPython.display import Markdown, display
!python scripts/run.py tables --results {OUT}
!python scripts/run.py bootstrap {COMMON}
display(Markdown(open(f"{OUT}/tables/main.md").read()))
display(Markdown(open(f"{OUT}/tables/bootstrap.md").read()))

In [ ]:
shutil.make_archive("/content/final_results_part4", "zip", OUT, ".")
from google.colab import files
files.download("/content/final_results_part4.zip")